<a href="https://colab.research.google.com/github/NigthDragon5000/nanoTabFM/blob/main/nanoTabFM_1809.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn import MultiheadAttention, Linear, LayerNorm


# ============================================================
# MAIN MODEL
# ============================================================

class MiniTABFM(nn.Module):

    def __init__(
        self,
        embedding_size: int,
        num_attention_heads: int,
        mlp_hidden_size: int,
        num_layers: int,
        icl_num_layers: int,
        num_outputs: int,
    ):
        super().__init__()

        # ----------------------------------------------------
        # ORIGINAL NANO-TABPFN COMPONENTS
        # ----------------------------------------------------

        self.feature_encoder = FeatureEncoder(
            embedding_size
        )

        self.target_encoder = TargetEncoder(
            embedding_size
        )

        # Original Mini-TabPFN transformer blocks
        self.transformer_blocks = nn.ModuleList()

        for _ in range(num_layers):
            self.transformer_blocks.append(
                TransformerEncoderLayer(
                    embedding_size=embedding_size,
                    nhead=num_attention_heads,
                    mlp_hidden_size=mlp_hidden_size,
                )
            )

        # ----------------------------------------------------
        # NEW: ROW CLS
        # ----------------------------------------------------

        self.row_cls = RowCLS(
            embedding_size=embedding_size,
            num_attention_heads=num_attention_heads,
        )

        # ----------------------------------------------------
        # NEW: ROW-LEVEL ICL TRANSFORMER
        # ----------------------------------------------------

        self.icl_transformer = ICLTransformer(
            embedding_size=embedding_size,
            num_attention_heads=num_attention_heads,
            mlp_hidden_size=mlp_hidden_size,
            num_layers=icl_num_layers,
        )

        # ----------------------------------------------------
        # ORIGINAL STYLE DECODER
        # ----------------------------------------------------

        self.decoder = Decoder(
            embedding_size=embedding_size,
            mlp_hidden_size=mlp_hidden_size,
            num_outputs=num_outputs,
        )


    def forward(
        self,
        src: tuple[torch.Tensor, torch.Tensor],
        train_test_split_index: int,
    ) -> torch.Tensor:

        x_src, y_src = src

        # ----------------------------------------------------
        # ORIGINAL TARGET SHAPE HANDLING
        # ----------------------------------------------------

        if len(y_src.shape) < len(x_src.shape):
            y_src = y_src.unsqueeze(-1)

        # B = batch
        # R = rows
        # C = columns
        # E = embedding size

        # ----------------------------------------------------
        # ORIGINAL FEATURE ENCODER
        # ----------------------------------------------------

        x_src = self.feature_encoder(
            x_src,
            train_test_split_index
        )

        num_rows = x_src.shape[1]

        # ----------------------------------------------------
        # ORIGINAL TARGET ENCODER
        # ----------------------------------------------------

        y_src = self.target_encoder(
            y_src,
            num_rows
        )

        # ----------------------------------------------------
        # ORIGINAL TABLE REPRESENTATION
        # ----------------------------------------------------

        src = torch.cat(
            [x_src, y_src],
            dim=2
        )

        # src:
        # [B, R, C, E]
        #
        # The last column is the target column.

        # ----------------------------------------------------
        # ORIGINAL MINI-TABPFN TRANSFORMER
        # ----------------------------------------------------

        for block in self.transformer_blocks:

            src = block(
                src,
                train_test_split_index=train_test_split_index
            )

        # At this point:
        #
        # src = [B, R, C, E]
        #
        # Column attention has happened.
        # Row attention has happened.
        # MLP has happened.
        #
        # This is still essentially the original Mini-TabPFN.


        # ====================================================
        # NEW PART 1: ROW CLS
        # ====================================================

        row_embeddings = self.row_cls(src)

        # [B, R, C, E]
        #
        #       CLS
        #        ↓
        # [feature1, feature2, ..., featureC]
        #
        # becomes
        #
        # [B, R, E]


        # ====================================================
        # NEW PART 2: ICL TRANSFORMER
        # ====================================================

        row_embeddings = self.icl_transformer(
            row_embeddings,
            train_test_split_index=train_test_split_index,
        )

        # [B, R, E]


        # ====================================================
        # PREDICTION
        # ====================================================

        # Only test rows are predicted.
        output = row_embeddings[
            :,
            train_test_split_index:,
            :
        ]

        # [B, num_test_rows, E]

        output = self.decoder(output)

        # [B, num_test_rows, num_outputs]

        return output


# ============================================================
# ORIGINAL FEATURE ENCODER
# ============================================================

class FeatureEncoder(nn.Module):

    def __init__(self, embedding_size: int):

        super().__init__()

        self.linear_layer = nn.Linear(
            1,
            embedding_size
        )


    def forward(
        self,
        x: torch.Tensor,
        train_test_split_index: int,
    ) -> torch.Tensor:

        # [B, R, C]
        x = x.unsqueeze(-1)

        # [B, R, C, 1]

        # Statistics calculated only from training rows.
        mean = torch.mean(
            x[:, :train_test_split_index],
            dim=1,
            keepdim=True,
        )

        std = torch.std(
            x[:, :train_test_split_index],
            dim=1,
            keepdim=True,
        ) + 1e-20

        x = (x - mean) / std

        x = torch.clip(
            x,
            min=-100,
            max=100,
        )

        return self.linear_layer(x)

        # [B, R, C, E]


# ============================================================
# ORIGINAL TARGET ENCODER
# ============================================================

class TargetEncoder(nn.Module):

    def __init__(self, embedding_size: int):

        super().__init__()

        self.linear_layer = nn.Linear(
            1,
            embedding_size
        )


    def forward(
        self,
        y_train: torch.Tensor,
        num_rows: int,
    ) -> torch.Tensor:

        # y_train:
        # [B, num_train_rows, 1]

        mean = torch.mean(
            y_train,
            dim=1,
            keepdim=True,
        )

        # Padding for test rows.
        padding = mean.repeat(
            1,
            num_rows - y_train.shape[1],
            1,
        )

        y = torch.cat(
            [y_train, padding],
            dim=1,
        )

        # [B, R, 1]

        y = y.unsqueeze(-1)

        # [B, R, 1, 1]

        return self.linear_layer(y)

        # [B, R, 1, E]


# ============================================================
# ORIGINAL MINI-TABPFN TRANSFORMER BLOCK
# ============================================================

class TransformerEncoderLayer(nn.Module):

    def __init__(
        self,
        embedding_size: int,
        nhead: int,
        mlp_hidden_size: int,
        layer_norm_eps: float = 1e-5,
        batch_first: bool = True,
    ):

        super().__init__()

        # ----------------------------------------------------
        # ORIGINAL COLUMN ATTENTION
        # ----------------------------------------------------

        self.self_attention_between_features = (
            MultiheadAttention(
                embedding_size,
                nhead,
                batch_first=batch_first,
            )
        )

        # ----------------------------------------------------
        # ORIGINAL ROW ATTENTION
        # ----------------------------------------------------

        self.self_attention_between_datapoints = (
            MultiheadAttention(
                embedding_size,
                nhead,
                batch_first=batch_first,
            )
        )

        # ----------------------------------------------------
        # ORIGINAL MLP
        # ----------------------------------------------------

        self.linear1 = Linear(
            embedding_size,
            mlp_hidden_size,
        )

        self.linear2 = Linear(
            mlp_hidden_size,
            embedding_size,
        )

        # ----------------------------------------------------
        # ORIGINAL NORMALIZATION
        # ----------------------------------------------------

        self.norm1 = LayerNorm(
            embedding_size,
            eps=layer_norm_eps,
        )

        self.norm2 = LayerNorm(
            embedding_size,
            eps=layer_norm_eps,
        )

        self.norm3 = LayerNorm(
            embedding_size,
            eps=layer_norm_eps,
        )


    def forward(
        self,
        src: torch.Tensor,
        train_test_split_index: int,
    ) -> torch.Tensor:

        # src:
        #
        # [B, R, C, E]

        batch_size, rows_size, col_size, embedding_size = src.shape

        # ====================================================
        # 1. ORIGINAL COLUMN ATTENTION
        # ====================================================

        src = src.reshape(
            batch_size * rows_size,
            col_size,
            embedding_size,
        )

        # [B*R, C, E]

        src = (
            self.self_attention_between_features(
                src,
                src,
                src,
            )[0]
            + src
        )

        # [B*R, C, E]

        src = src.reshape(
            batch_size,
            rows_size,
            col_size,
            embedding_size,
        )

        # [B, R, C, E]

        src = self.norm1(src)


        # ====================================================
        # 2. ORIGINAL ROW ATTENTION
        # ====================================================

        # Move columns before rows.

        src = src.transpose(1, 2)

        # [B, C, R, E]

        src = src.reshape(
            batch_size * col_size,
            rows_size,
            embedding_size,
        )

        # [B*C, R, E]


        # ----------------------------------------------------
        # TRAINING ROWS ATTEND TO TRAINING ROWS
        # ----------------------------------------------------

        src_left = self.self_attention_between_datapoints(
            src[:, :train_test_split_index],
            src[:, :train_test_split_index],
            src[:, :train_test_split_index],
        )[0]


        # ----------------------------------------------------
        # TEST ROWS ATTEND TO TRAINING ROWS
        # ----------------------------------------------------

        src_right = self.self_attention_between_datapoints(
            src[:, train_test_split_index:],
            src[:, :train_test_split_index],
            src[:, :train_test_split_index],
        )[0]


        # Put them back together.

        src = torch.cat(
            [
                src_left,
                src_right,
            ],
            dim=1,
        ) + src

        # [B*C, R, E]

        src = src.reshape(
            batch_size,
            col_size,
            rows_size,
            embedding_size,
        )

        # [B, C, R, E]

        src = src.transpose(2, 1)

        # [B, R, C, E]

        src = self.norm2(src)


        # ====================================================
        # 3. ORIGINAL MLP
        # ====================================================

        src = (
            self.linear2(
                F.gelu(
                    self.linear1(src)
                )
            )
            + src
        )

        src = self.norm3(src)

        return src


# ============================================================
# NEW: ROW CLS
# ============================================================

class RowCLS(nn.Module):

    """
    Collapses the C column embeddings of each row
    into a single row embedding.

    Input:
        [B, R, C, E]

    Output:
        [B, R, E]
    """

    def __init__(
        self,
        embedding_size: int,
        num_attention_heads: int,
    ):

        super().__init__()

        # One learned CLS query.

        self.cls_token = nn.Parameter(
            torch.randn(
                1,
                1,
                embedding_size,
            ) * 0.02
        )

        self.attention = MultiheadAttention(
            embedding_size,
            num_attention_heads,
            batch_first=True,
        )

        self.norm = LayerNorm(
            embedding_size
        )


    def forward(
        self,
        src: torch.Tensor,
    ) -> torch.Tensor:

        # src:
        # [B, R, C, E]

        B, R, C, E = src.shape

        # Treat every row as an independent sequence.

        src = src.reshape(
            B * R,
            C,
            E,
        )

        # [B*R, C, E]

        # Create one CLS query for every row.

        cls = self.cls_token.expand(
            B * R,
            -1,
            -1,
        )

        # [B*R, 1, E]

        # ----------------------------------------------------
        # CLS attends to the C columns
        # ----------------------------------------------------

        cls_output = self.attention(
            query=cls,
            key=src,
            value=src,
        )[0]

        # [B*R, 1, E]

        cls_output = self.norm(
            cls_output
        )

        # Remove sequence dimension.

        cls_output = cls_output.squeeze(1)

        # [B*R, E]

        # Restore row structure.

        cls_output = cls_output.reshape(
            B,
            R,
            E,
        )

        # [B, R, E]

        return cls_output


# ============================================================
# NEW: ROW-LEVEL ICL TRANSFORMER
# ============================================================

class ICLTransformer(nn.Module):

    """
    Operates after the Row CLS compression.

    Input:
        [B, R, E]

    The training rows interact with training rows,
    while test rows attend to the training rows.

    Output:
        [B, R, E]
    """

    def __init__(
        self,
        embedding_size: int,
        num_attention_heads: int,
        mlp_hidden_size: int,
        num_layers: int,
    ):

        super().__init__()

        self.layers = nn.ModuleList(
            [
                ICLTransformerLayer(
                    embedding_size=embedding_size,
                    nhead=num_attention_heads,
                    mlp_hidden_size=mlp_hidden_size,
                )
                for _ in range(num_layers)
            ]
        )


    def forward(
        self,
        src: torch.Tensor,
        train_test_split_index: int,
    ) -> torch.Tensor:

        # [B, R, E]

        for layer in self.layers:

            src = layer(
                src,
                train_test_split_index,
            )

        return src


# ============================================================
# NEW: SINGLE ICL TRANSFORMER LAYER
# ============================================================

class ICLTransformerLayer(nn.Module):

    def __init__(
        self,
        embedding_size: int,
        nhead: int,
        mlp_hidden_size: int,
    ):

        super().__init__()

        self.attention = MultiheadAttention(
            embedding_size,
            nhead,
            batch_first=True,
        )

        self.linear1 = Linear(
            embedding_size,
            mlp_hidden_size,
        )

        self.linear2 = Linear(
            mlp_hidden_size,
            embedding_size,
        )

        self.norm1 = LayerNorm(
            embedding_size
        )

        self.norm2 = LayerNorm(
            embedding_size
        )


    def forward(
        self,
        src: torch.Tensor,
        train_test_split_index: int,
    ):

        # src:
        # [B, R, E]

        # ----------------------------------------------------
        # TRAINING ROWS ATTEND TO TRAINING ROWS
        # ----------------------------------------------------

        train = src[
            :,
            :train_test_split_index,
            :
        ]

        # [B, N_train, E]

        train_out = self.attention(
            query=train,
            key=train,
            value=train,
        )[0]

        train_out = train + train_out

        train_out = self.norm1(
            train_out
        )


        # ----------------------------------------------------
        # TEST ROWS ATTEND TO TRAINING ROWS
        # ----------------------------------------------------

        test = src[
            :,
            train_test_split_index:,
            :
        ]

        # [B, N_test, E]

        test_out = self.attention(
            query=test,
            key=train,
            value=train,
        )[0]

        test_out = test + test_out

        test_out = self.norm1(
            test_out
        )


        # ----------------------------------------------------
        # RESTORE FULL SEQUENCE
        # ----------------------------------------------------

        src = torch.cat(
            [
                train_out,
                test_out,
            ],
            dim=1,
        )

        # [B, R, E]


        # ----------------------------------------------------
        # MLP
        # ----------------------------------------------------

        src = src + self.linear2(
            F.gelu(
                self.linear1(src)
            )
        )

        src = self.norm2(src)

        return src


# ============================================================
# ORIGINAL DECODER
# ============================================================

class Decoder(nn.Module):

    def __init__(
        self,
        embedding_size: int,
        mlp_hidden_size: int,
        num_outputs: int,
    ):

        super().__init__()

        self.linear1 = nn.Linear(
            embedding_size,
            mlp_hidden_size,
        )

        self.linear2 = nn.Linear(
            mlp_hidden_size,
            num_outputs,
        )


    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:

        return self.linear2(
            F.gelu(
                self.linear1(x)
            )
        )

In [3]:
# ============================================================
# IMPORTS
# ============================================================

import copy
import numpy as np
import torch
from torch import nn

from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# SYNTHETIC DATA POOL
# ============================================================

def generate_pool(
    n_samples,
    n_features=20,
    random_state=42,
):

    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,

        n_informative=10,
        n_redundant=5,
        n_repeated=0,

        n_classes=2,

        # Some overlap makes the problem less trivial.
        class_sep=1.0,

        # A little label noise.
        flip_y=0.03,

        # More interesting class structure.
        n_clusters_per_class=2,

        random_state=random_state,
    )

    X = X.astype(np.float32)
    y = y.astype(np.float32)

    return (
        torch.from_numpy(X),
        torch.from_numpy(y),
    )


# ============================================================
# CREATE THREE COMPLETELY SEPARATE DATA POOLS
# ============================================================

X_train_pool, y_train_pool = generate_pool(
    n_samples=50_000,
    n_features=20,
    random_state=SEED,
)

X_val_pool, y_val_pool = generate_pool(
    n_samples=10_000,
    n_features=20,
    random_state=1234,
)

X_test_pool, y_test_pool = generate_pool(
    n_samples=10_000,
    n_features=20,
    random_state=9999,
)


print("\nData pools:")
print("Train:", X_train_pool.shape)
print("Validation:", X_val_pool.shape)
print("Test:", X_test_pool.shape)


# ============================================================
# EPISODE SAMPLER
# ============================================================

def sample_episode_batch(
    X_pool,
    y_pool,
    batch_size,
    context_size,
    device,
):
    """
    Generate a batch of independent ICL episodes.

    Each episode contains:

        context_size context rows
        +
        1 query row

    Returns:

        X_episode
            [B, context_size + 1, C]

        y_context
            [B, context_size, 1]

        y_query
            [B]
    """

    n_pool = X_pool.shape[0]

    total_rows = context_size + 1

    # --------------------------------------------------------
    # Randomly sample rows for each episode.
    #
    # Sampling with replacement is intentional here.
    # The pool is much larger than an episode, so collisions
    # are uncommon and this keeps generation very cheap.
    # --------------------------------------------------------

    indices = torch.randint(
        low=0,
        high=n_pool,
        size=(batch_size, total_rows),
    )

    X_episode = X_pool[
        indices
    ]

    y_episode = y_pool[
        indices
    ]

    # --------------------------------------------------------
    # Last row is ALWAYS the query.
    # --------------------------------------------------------

    y_context = y_episode[
        :, :-1
    ].unsqueeze(-1)

    y_query = y_episode[
        :, -1
    ]

    return (
        X_episode.to(device),
        y_context.to(device),
        y_query.to(device),
    )


# ============================================================
# CREATE MODEL
# ============================================================

model = MiniTABFM(
    embedding_size=64,
    num_attention_heads=4,
    mlp_hidden_size=128,

    # Original Mini-TabPFN blocks
    num_layers=3,

    # Our new row-level ICL Transformer
    icl_num_layers=2,

    # Binary classification
    num_outputs=1,
).to(device)


num_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "\nModel parameters:",
    f"{num_parameters:,}",
    f"({num_parameters / 1e6:.3f} M)"
)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)


# ============================================================
# LOSS
# ============================================================

criterion = nn.BCEWithLogitsLoss()


# ============================================================
# TRAINING STEP
# ============================================================

def train_step(
    model,
    X_pool,
    y_pool,
    optimizer,
    criterion,
    batch_size,
    context_size,
):

    model.train()

    # --------------------------------------------------------
    # Generate independent episodes
    # --------------------------------------------------------

    X_episode, y_context, y_query = (
        sample_episode_batch(
            X_pool=X_pool,
            y_pool=y_pool,
            batch_size=batch_size,
            context_size=context_size,
            device=device,
        )
    )

    # --------------------------------------------------------
    # Forward pass
    # --------------------------------------------------------

    logits = model(
        (
            X_episode,
            y_context,
        ),
        train_test_split_index=context_size,
    )

    # Expected:
    #
    # [B, 1, 1]
    #
    # One prediction for every query.

    query_logits = logits[
        :, 0, 0
    ]

    # [B]

    loss = criterion(
        query_logits,
        y_query,
    )

    # --------------------------------------------------------
    # Backpropagation
    # --------------------------------------------------------

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    optimizer.step()

    return loss.item()


# ============================================================
# EVALUATION ON MANY INDEPENDENT EPISODES
# ============================================================

@torch.no_grad()
def evaluate_auc(
    model,
    X_pool,
    y_pool,
    num_episodes,
    batch_size,
    context_size,
):

    model.eval()

    all_predictions = []
    all_targets = []

    episodes_done = 0

    while episodes_done < num_episodes:

        current_batch_size = min(
            batch_size,
            num_episodes - episodes_done,
        )

        # ----------------------------------------------------
        # New independent episodes
        # ----------------------------------------------------

        X_episode, y_context, y_query = (
            sample_episode_batch(
                X_pool=X_pool,
                y_pool=y_pool,
                batch_size=current_batch_size,
                context_size=context_size,
                device=device,
            )
        )

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        logits = model(
            (
                X_episode,
                y_context,
            ),
            train_test_split_index=context_size,
        )

        query_logits = logits[
            :, 0, 0
        ]

        probabilities = torch.sigmoid(
            query_logits
        )

        all_predictions.extend(
            probabilities.cpu().numpy()
        )

        all_targets.extend(
            y_query.cpu().numpy()
        )

        episodes_done += current_batch_size

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Number of AUC observations =
    # number of episodes
    # --------------------------------------------------------

    auc = roc_auc_score(
        all_targets,
        all_predictions,
    )

    return auc


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

CONTEXT_SIZE = 64

BATCH_EPISODES = 32

TRAIN_STEPS = 2_000

VALIDATION_EPISODES = 1_000

TEST_EPISODES = 5_000

EVAL_EVERY = 100


print("\nTraining configuration:")
print("Context rows:", CONTEXT_SIZE)
print("Episodes per batch:", BATCH_EPISODES)
print("Training steps:", TRAIN_STEPS)
print("Validation episodes:", VALIDATION_EPISODES)
print("Final test episodes:", TEST_EPISODES)


# ============================================================
# TRAINING LOOP
# ============================================================

best_val_auc = -np.inf
best_state = None

running_loss = 0.0

for step in range(
    1,
    TRAIN_STEPS + 1,
):

    loss = train_step(
        model=model,
        X_pool=X_train_pool,
        y_pool=y_train_pool,
        optimizer=optimizer,
        criterion=criterion,
        batch_size=BATCH_EPISODES,
        context_size=CONTEXT_SIZE,
    )

    running_loss += loss

    # --------------------------------------------------------
    # Periodic validation
    # --------------------------------------------------------

    if (
        step % EVAL_EVERY == 0
        or step == 1
    ):

        mean_loss = (
            running_loss / EVAL_EVERY
            if step > 1
            else running_loss
        )

        val_auc = evaluate_auc(
            model=model,
            X_pool=X_val_pool,
            y_pool=y_val_pool,
            num_episodes=VALIDATION_EPISODES,
            batch_size=BATCH_EPISODES,
            context_size=CONTEXT_SIZE,
        )

        print(
            f"Step {step:04d} | "
            f"Loss: {mean_loss:.4f} | "
            f"Validation AUC: {val_auc:.4f}"
        )

        # ----------------------------------------------------
        # Save best model
        # ----------------------------------------------------

        if val_auc > best_val_auc:

            best_val_auc = val_auc

            best_state = copy.deepcopy(
                model.state_dict()
            )

        running_loss = 0.0


# ============================================================
# RESTORE BEST MODEL
# ============================================================

if best_state is not None:

    model.load_state_dict(
        best_state
    )

print(
    "\nBest validation AUC:",
    f"{best_val_auc:.4f}"
)


# ============================================================
# FINAL TEST
# ============================================================

test_auc = evaluate_auc(
    model=model,
    X_pool=X_test_pool,
    y_pool=y_test_pool,
    num_episodes=TEST_EPISODES,
    batch_size=BATCH_EPISODES,
    context_size=CONTEXT_SIZE,
)


print("\n" + "=" * 60)
print(
    f"FINAL UNSEEN TEST AUC: {test_auc:.4f}"
)
print("=" * 60)

print(
    f"\nThe final AUC is based on "
    f"{TEST_EPISODES:,} independent query predictions."
)

Device: cuda

Data pools:
Train: torch.Size([50000, 20])
Validation: torch.Size([10000, 20])
Test: torch.Size([10000, 20])

Model parameters: 243,201 (0.243 M)

Training configuration:
Context rows: 64
Episodes per batch: 32
Training steps: 2000
Validation episodes: 1000
Final test episodes: 5000
Step 0001 | Loss: 0.6986 | Validation AUC: 0.6485
Step 0100 | Loss: 0.6800 | Validation AUC: 0.6705
Step 0200 | Loss: 0.6697 | Validation AUC: 0.6575
Step 0300 | Loss: 0.6707 | Validation AUC: 0.6730
Step 0400 | Loss: 0.6588 | Validation AUC: 0.6767
Step 0500 | Loss: 0.6609 | Validation AUC: 0.6870
Step 0600 | Loss: 0.6540 | Validation AUC: 0.6758
Step 0700 | Loss: 0.6581 | Validation AUC: 0.6690
Step 0800 | Loss: 0.6124 | Validation AUC: 0.8726
Step 0900 | Loss: 0.5446 | Validation AUC: 0.8731
Step 1000 | Loss: 0.5277 | Validation AUC: 0.8717
Step 1100 | Loss: 0.5182 | Validation AUC: 0.8837
Step 1200 | Loss: 0.5184 | Validation AUC: 0.8656
Step 1300 | Loss: 0.5075 | Validation AUC: 0.8856
St